# 05 — Map Analysis: How Map Mode Shapes Strategy and Outcomes

Overwatch maps are divided into distinct modes, each with different objective mechanics that favor different strategies:

> **"Map mode affects strategy; attacker/defender advantages vary significantly across maps."**
> — Standard coaching knowledge

### Key OW Concepts
- **Map Modes**: Control (KOTH — best of 3 rounds), Escort (payload push), Hybrid (capture point then push payload), Push (robot push, OW2-specific), Flashpoint (capture rotating points).
- **Attacker vs Defender**: On Escort and Hybrid, one team attacks (pushes payload/captures point) while the other defends. Roles swap between rounds.
- **Map geometry**: Sightlines, choke points, verticality, and flank routes vary dramatically across maps and dictate which heroes/comps thrive.
- **Momentum**: In multi-round formats, winning the first round can create psychological and strategic momentum.

### Analysis Plan
1. Win rates by map and map mode
2. Attacker vs defender analysis (Hybrid/Escort)
3. Round-by-round momentum analysis
4. Map-composition preferences
5. Coaching implications

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data_loader import load_csv, load_matches, load_rounds, load_player_stats
from src.preprocessing import (
    determine_match_winner, add_role_column, HERO_ROLES,
    classify_composition
)
from src.visualization import setup_style, OW_COLORS, OW_PALETTE, ROLE_COLORS, save_fig, role_color

setup_style()
pd.set_option('display.max_columns', 30)

## 1. Load Data

In [ ]:
match_start, match_end = load_matches()
round_start, round_end = load_rounds()
player_stats = load_player_stats()
matches = determine_match_winner(match_end, match_start)

print(f"Matches:     {len(matches):,}")
print(f"RoundStart:  {len(round_start):,}")
print(f"RoundEnd:    {len(round_end):,}")
print(f"PlayerStats: {len(player_stats):,}")
print()
print("Map types:", matches['map_type'].value_counts().to_dict())
print(f"Unique maps: {matches['map_name'].nunique()}")

In [ ]:
# Quick overview of maps
map_overview = matches.groupby(['map_type', 'map_name']).agg(
    matches=('MapDataId', 'nunique'),
    draws=('winner', lambda x: (x == 'Draw').sum())
).reset_index()
map_overview['non_draw'] = map_overview['matches'] - map_overview['draws']
map_overview = map_overview.sort_values(['map_type', 'matches'], ascending=[True, False])

print("Maps in dataset:")
print(map_overview.to_string(index=False))

## 2. Win Rates by Map and Map Mode

Are some maps significantly more balanced than others? Do certain map modes produce more draws or one-sided results?

In [ ]:
# For win rate analysis, we look at team_1 win rate (since team assignment is arbitrary,
# this should hover around 50% for balanced maps)
matches_valid = matches[matches['winner'] != 'Draw'].copy()
matches_valid['team_1_won'] = matches_valid['winner'] == matches_valid['team_1_name']

# Win rate by map mode
mode_stats = matches_valid.groupby('map_type').agg(
    total=('team_1_won', 'count'),
    team_1_wins=('team_1_won', 'sum')
)
mode_stats['team_1_wr'] = mode_stats['team_1_wins'] / mode_stats['total'] * 100

print("Team 1 Win Rate by Map Mode (should be ~50% if balanced):")
print(mode_stats.to_string())
print()

# Draw rates by mode (using all matches including draws)
draw_rates = matches.groupby('map_type').agg(
    total=('winner', 'count'),
    draws=('winner', lambda x: (x == 'Draw').sum())
)
draw_rates['draw_rate'] = draw_rates['draws'] / draw_rates['total'] * 100
print("Draw Rate by Map Mode:")
print(draw_rates.to_string())

In [ ]:
# Map-level score differentials (closeness of matches)
matches_scored = matches.copy()
matches_scored['score_diff'] = abs(matches_scored['team_1_score'] - matches_scored['team_2_score'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Draw rate by mode
draw_rates_sorted = draw_rates.sort_values('draw_rate', ascending=True)
axes[0].barh(draw_rates_sorted.index, draw_rates_sorted['draw_rate'],
             color=OW_COLORS['orange'], alpha=0.85)
for i, (mode, row) in enumerate(draw_rates_sorted.iterrows()):
    axes[0].text(row['draw_rate'] + 0.3, i,
                 f'{row["draw_rate"]:.1f}% (n={row["total"]:,})',
                 va='center', fontsize=10, color=OW_COLORS['white'])
axes[0].set_xlabel('Draw Rate (%)')
axes[0].set_title('Draw Rate by Map Mode')

# Score differential by mode
mode_order = matches_scored.groupby('map_type')['score_diff'].median().sort_values().index
bp = axes[1].boxplot(
    [matches_scored[matches_scored['map_type'] == m]['score_diff'] for m in mode_order],
    labels=mode_order,
    patch_artist=True,
    showfliers=False,
    medianprops={'color': OW_COLORS['gold'], 'linewidth': 2}
)
for patch in bp['boxes']:
    patch.set_facecolor(OW_COLORS['blue'])
    patch.set_alpha(0.7)
axes[1].set_ylabel('Score Differential')
axes[1].set_title('Match Closeness by Map Mode (Score Differential)')

plt.tight_layout()
save_fig(fig, '05_map_mode_balance')
plt.show()

In [ ]:
# Win rate by individual map (team_1 win rate as balance indicator)
map_wr = matches_valid.groupby('map_name').agg(
    total=('team_1_won', 'count'),
    team_1_wins=('team_1_won', 'sum'),
    map_type=('map_type', 'first')
)
map_wr['team_1_wr'] = map_wr['team_1_wins'] / map_wr['total'] * 100
map_wr = map_wr[map_wr['total'] >= 10]  # Minimum matches
map_wr = map_wr.sort_values('team_1_wr')

# Color by how far from 50%
fig, ax = plt.subplots(figsize=(14, 10))

# Color maps by mode
mode_colors = {
    'Control': OW_COLORS['blue'],
    'Escort': OW_COLORS['orange'],
    'Hybrid': OW_COLORS['green'],
    'Push': OW_COLORS['teal'],
    'Flashpoint': OW_COLORS['purple'],
}
colors = [mode_colors.get(mt, OW_COLORS['light_gray']) for mt in map_wr['map_type']]

bars = ax.barh(map_wr.index, map_wr['team_1_wr'], color=colors, alpha=0.85)
ax.axvline(50, color=OW_COLORS['gold'], linestyle='--', linewidth=2, label='Balanced (50%)')

for bar, (map_name, row) in zip(bars, map_wr.iterrows()):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'n={row["total"]}', va='center', fontsize=8, color=OW_COLORS['light_gray'])

ax.set_xlabel('Team 1 Win Rate (%)')
ax.set_title('Map Balance: Team 1 Win Rate by Map\n(Closer to 50% = more balanced)')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=m) for m, c in mode_colors.items()
                   if m in map_wr['map_type'].values]
legend_elements.append(plt.Line2D([0], [0], color=OW_COLORS['gold'], linestyle='--', label='Balanced (50%)'))
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
save_fig(fig, '05_map_balance_chart')
plt.show()

## 3. Attacker vs Defender Analysis

On Escort and Hybrid maps, teams take turns attacking and defending. We use RoundEnd scores to understand whether attack or defense has an inherent advantage.

In the Parsertime data, `RoundEnd` contains `team_1_score` and `team_2_score` at the end of each round, plus `capturing_team` which indicates who was attacking (pushing the objective).

In [ ]:
# Enrich round data with map info
rounds = round_end.merge(
    match_start[['MapDataId', 'map_name', 'map_type', 'team_1_name', 'team_2_name']],
    on='MapDataId',
    how='inner'
)

# Also join with match winner
rounds = rounds.merge(
    matches[['MapDataId', 'winner']],
    on='MapDataId',
    how='left'
)

print(f"Rounds with map info: {len(rounds):,}")
print(f"Rounds by map type:")
print(rounds['map_type'].value_counts())
print()
print("RoundEnd columns:", list(round_end.columns))

In [ ]:
# Analyze score progression by round for attack/defense modes
# For Escort and Hybrid, the capturing_team is the attacker
attack_modes = ['Escort', 'Hybrid']
attack_rounds = rounds[rounds['map_type'].isin(attack_modes)].copy()

if len(attack_rounds) > 0 and 'capturing_team' in attack_rounds.columns:
    # Determine if the capturing (attacking) team won the match
    attack_rounds['attacker_won_match'] = attack_rounds['capturing_team'] == attack_rounds['winner']
    
    # Score gained per round by attacking team
    attack_rounds = attack_rounds.sort_values(['MapDataId', 'round_number'])
    
    print(f"Attack/defense rounds analyzed: {len(attack_rounds):,}")
    print(f"Maps with attack/defense: {attack_rounds['map_name'].nunique()}")
    
    # Win rate for attacking vs defending teams across matches
    attack_win = attack_rounds.groupby('MapDataId').agg(
        attacker_won=('attacker_won_match', 'first'),
        map_name=('map_name', 'first'),
        map_type=('map_type', 'first')
    )
    
    attack_wr_by_map = attack_win.groupby('map_name').agg(
        total=('attacker_won', 'count'),
        attacker_wins=('attacker_won', 'sum'),
        map_type=('map_type', 'first')
    )
    attack_wr_by_map['attacker_wr'] = attack_wr_by_map['attacker_wins'] / attack_wr_by_map['total'] * 100
    attack_wr_by_map = attack_wr_by_map[attack_wr_by_map['total'] >= 5]
    attack_wr_by_map = attack_wr_by_map.sort_values('attacker_wr')
    
    print("\nAttacker win rate by map:")
    print(attack_wr_by_map[['total', 'attacker_wr', 'map_type']].to_string())
else:
    print("No attack/defense round data available or 'capturing_team' column not present.")
    print("Skipping attacker/defender analysis.")

In [ ]:
# Visualization: Attacker advantage by map (if data available)
if len(attack_rounds) > 0 and 'capturing_team' in attack_rounds.columns and len(attack_wr_by_map) > 0:
    fig, ax = plt.subplots(figsize=(12, 8))
    
    colors = [mode_colors.get(mt, OW_COLORS['light_gray']) for mt in attack_wr_by_map['map_type']]
    bars = ax.barh(attack_wr_by_map.index, attack_wr_by_map['attacker_wr'],
                   color=colors, alpha=0.85)
    
    ax.axvline(50, color=OW_COLORS['gold'], linestyle='--', linewidth=2, label='Even (50%)')
    
    for bar, (_, row) in zip(bars, attack_wr_by_map.iterrows()):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{row["attacker_wr"]:.0f}% (n={row["total"]})',
                va='center', fontsize=9, color=OW_COLORS['white'])
    
    ax.set_xlabel('Attacker Win Rate (%)')
    ax.set_title('Attacker vs Defender Advantage by Map')
    ax.legend()
    
    plt.tight_layout()
    save_fig(fig, '05_attacker_defender_advantage')
    plt.show()
else:
    print("Skipping attacker/defender visualization due to insufficient data.")

## 4. Round-by-Round Momentum Analysis

Does winning the first round predict winning the match? Momentum is a significant factor in competitive play — winning early can tilt the mental game.

In [ ]:
# Analyze first-round results vs match outcomes
# Get score at end of round 1 for each match
first_rounds = rounds[rounds['round_number'] == 1].copy()
first_rounds = first_rounds[first_rounds['winner'] != 'Draw']

# Determine round 1 leader
def round_leader(row):
    if row['team_1_score'] > row['team_2_score']:
        return row['team_1_name']
    elif row['team_2_score'] > row['team_1_score']:
        return row['team_2_name']
    return 'Tied'

first_rounds['round_1_leader'] = first_rounds.apply(round_leader, axis=1)
first_rounds['r1_leader_won_match'] = first_rounds['round_1_leader'] == first_rounds['winner']

# Filter out ties in round 1
decisive_r1 = first_rounds[first_rounds['round_1_leader'] != 'Tied']

r1_win_rate = decisive_r1['r1_leader_won_match'].mean() * 100
n_decisive = len(decisive_r1)

print(f"Round 1 Momentum Analysis")
print("=" * 40)
print(f"Matches with decisive Round 1: {n_decisive:,}")
print(f"Round 1 leader won match: {r1_win_rate:.1f}%")
print()

# Break down by map mode
r1_by_mode = decisive_r1.groupby('map_type').agg(
    total=('r1_leader_won_match', 'count'),
    wins=('r1_leader_won_match', 'sum')
)
r1_by_mode['win_rate'] = r1_by_mode['wins'] / r1_by_mode['total'] * 100
print("Round 1 momentum by map mode:")
print(r1_by_mode.to_string())

In [ ]:
# Visualization: Round 1 momentum
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall
categories = ['R1 Leader\nWins Match', 'R1 Leader\nLoses Match']
values = [r1_win_rate, 100 - r1_win_rate]
bar_colors = [OW_COLORS['green'], OW_COLORS['red']]

bars = axes[0].bar(categories, values, color=bar_colors, width=0.5,
                   edgecolor=OW_COLORS['dark_blue'])
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontsize=14, fontweight='bold',
                 color=OW_COLORS['white'])
axes[0].set_ylabel('Percentage')
axes[0].set_title(f'Does Winning Round 1 Predict Match Win?\n(n={n_decisive:,} matches)')
axes[0].set_ylim(0, 100)
axes[0].axhline(50, color=OW_COLORS['gold'], linestyle='--', alpha=0.5)

# By mode
r1_by_mode_sorted = r1_by_mode.sort_values('win_rate')
axes[1].barh(r1_by_mode_sorted.index, r1_by_mode_sorted['win_rate'],
             color=OW_COLORS['orange'], alpha=0.85)
for i, (mode, row) in enumerate(r1_by_mode_sorted.iterrows()):
    axes[1].text(row['win_rate'] + 0.5, i,
                 f'{row["win_rate"]:.1f}% (n={row["total"]})',
                 va='center', fontsize=10, color=OW_COLORS['white'])
axes[1].axvline(50, color=OW_COLORS['gold'], linestyle='--', alpha=0.7)
axes[1].set_xlabel('R1 Leader Match Win Rate (%)')
axes[1].set_title('Round 1 Momentum by Map Mode')

plt.tight_layout()
save_fig(fig, '05_round1_momentum')
plt.show()

In [ ]:
# Score progression across rounds
# Track cumulative score advantage for the eventual winner across rounds
round_progression = rounds.copy()
round_progression = round_progression[round_progression['winner'] != 'Draw']

# Calculate winner's score advantage at end of each round
def winner_advantage(row):
    if row['winner'] == row['team_1_name']:
        return row['team_1_score'] - row['team_2_score']
    else:
        return row['team_2_score'] - row['team_1_score']

round_progression['winner_advantage'] = round_progression.apply(winner_advantage, axis=1)

# Average winner advantage by round number
round_avg = round_progression.groupby('round_number')['winner_advantage'].agg(['mean', 'count', 'std'])
round_avg = round_avg[round_avg['count'] >= 20]  # Minimum sample

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(round_avg.index, round_avg['mean'], 'o-', color=OW_COLORS['orange'],
        linewidth=2, markersize=8)
ax.fill_between(round_avg.index,
                round_avg['mean'] - round_avg['std'] * 0.5,
                round_avg['mean'] + round_avg['std'] * 0.5,
                alpha=0.2, color=OW_COLORS['orange'])

ax.axhline(0, color=OW_COLORS['gold'], linestyle='--', alpha=0.7, label='Even')
ax.set_xlabel('Round Number')
ax.set_ylabel('Score Advantage (Winner - Loser)')
ax.set_title('How Early Do Match Winners Take the Lead?\n(Average score advantage by round)')
ax.legend()

# Add sample size annotations
for rnd, row in round_avg.iterrows():
    ax.text(rnd, row['mean'] + 0.15, f'n={row["count"]:,}',
            ha='center', fontsize=8, color=OW_COLORS['light_gray'])

plt.tight_layout()
save_fig(fig, '05_score_progression')
plt.show()

## 5. Map-Composition Preferences

Which compositions are favored on which maps? We combine the composition classification from Notebook 03 with map data to see if certain archetypes perform better on certain maps.

In [ ]:
# Get player compositions per match+team using PlayerStat
ps = player_stats.merge(
    matches[['MapDataId', 'winner', 'map_name', 'map_type']],
    on='MapDataId',
    how='inner'
)
ps = ps[ps['hero_time_played'] >= 60].copy()
ps['team_won'] = ps['player_team'] == ps['winner']

# Primary hero per player per match (most time played)
primary = ps.sort_values('hero_time_played', ascending=False).groupby(
    ['MapDataId', 'player_team', 'player_name']
).first().reset_index()

# Classify team comp
team_comps = primary.groupby(['MapDataId', 'player_team']).agg(
    heroes=('player_hero', list),
    team_won=('team_won', 'first'),
    map_name=('map_name', 'first'),
    map_type=('map_type', 'first')
).reset_index()

team_comps['archetype'] = team_comps['heroes'].apply(classify_composition)

print(f"Team-match compositions: {len(team_comps):,}")
print(f"Archetype distribution:\n{team_comps['archetype'].value_counts()}")

In [ ]:
# Comp win rate by map mode
comp_map_mode = team_comps.groupby(['map_type', 'archetype']).agg(
    total=('team_won', 'count'),
    wins=('team_won', 'sum')
)
comp_map_mode['win_rate'] = comp_map_mode['wins'] / comp_map_mode['total'] * 100
comp_map_mode = comp_map_mode.reset_index()

# Pivot for heatmap
comp_mode_pivot = comp_map_mode.pivot_table(
    index='map_type', columns='archetype', values='win_rate'
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    comp_mode_pivot,
    annot=True, fmt='.1f', cmap='RdYlGn', center=50,
    vmin=35, vmax=65,
    linewidths=0.5, linecolor=OW_COLORS['dark_blue'],
    ax=ax,
    cbar_kws={'label': 'Win Rate (%)'}
)
ax.set_title('Composition Win Rate by Map Mode')
ax.set_ylabel('Map Mode')
ax.set_xlabel('Composition Archetype')

plt.tight_layout()
save_fig(fig, '05_comp_win_rate_by_mode')
plt.show()

In [ ]:
# Comp distribution by map (top maps)
top_maps = matches.groupby('map_name').size().nlargest(10).index.tolist()

comp_map = team_comps[team_comps['map_name'].isin(top_maps)].groupby(
    ['map_name', 'archetype']
).size().reset_index(name='count')

# Normalize to percentages per map
map_totals = comp_map.groupby('map_name')['count'].sum()
comp_map['pct'] = comp_map.apply(lambda r: r['count'] / map_totals[r['map_name']] * 100, axis=1)

comp_map_pivot = comp_map.pivot_table(
    index='map_name', columns='archetype', values='pct', fill_value=0
)

fig, ax = plt.subplots(figsize=(12, 8))
comp_map_pivot.plot(kind='barh', stacked=True, ax=ax,
                    color=[OW_COLORS['teal'], OW_COLORS['orange'],
                           OW_COLORS['light_gray'], OW_COLORS['blue']],
                    edgecolor=OW_COLORS['dark_blue'])
ax.set_xlabel('Composition Distribution (%)')
ax.set_title('Which Compositions Are Played on Each Map?')
ax.legend(title='Archetype', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
save_fig(fig, '05_comp_distribution_by_map')
plt.show()

In [ ]:
# Hero preferences by map — which heroes have the most map-specific pick rate variation?
ps_with_role = add_role_column(ps)
map_hero_stats = ps_with_role.groupby(['map_name', 'player_hero']).agg(
    count=('MapDataId', 'count'),
    wins=('team_won', 'sum'),
    role=('role', 'first')
).reset_index()
map_hero_stats['win_rate'] = map_hero_stats['wins'] / map_hero_stats['count'] * 100

# Find heroes with highest win rate variance across maps
hero_map_variance = map_hero_stats[map_hero_stats['count'] >= 10].groupby('player_hero').agg(
    wr_std=('win_rate', 'std'),
    wr_range=('win_rate', lambda x: x.max() - x.min()),
    maps_played=('map_name', 'nunique'),
    total_picks=('count', 'sum'),
    role=('role', 'first')
)
hero_map_variance = hero_map_variance[hero_map_variance['maps_played'] >= 3]
hero_map_variance = hero_map_variance.sort_values('wr_range', ascending=False)

print("Heroes with highest map-dependent win rate variation:")
print("(Higher range = more map-dependent)")
print(hero_map_variance.head(15)[['wr_range', 'wr_std', 'maps_played', 'role']].to_string())

## 6. Summary & Coaching Implications

### Key Findings

| Topic | Finding |
|-------|---------|
| Map balance | See Team 1 win rate chart |
| Draw rates | Vary by mode (see above) |
| Attacker advantage | Map-specific (see analysis) |
| Round 1 momentum | Winning R1 predicts match outcome |
| Map-comp fit | Clear patterns exist |

### Coaching Implications

1. **Map veto/pick matters**: Not all maps are created equal. Some maps have significant structural advantages that favor certain playstyles. Teams should veto maps that do not suit their comp pool.

2. **First round is critical**: The momentum data shows that winning the first round significantly increases your chance of winning the match. Teams should put extra preparation into their opening round strategies.

3. **Map-specific comp prep**: The composition and hero pick rate data show clear map preferences. Teams should practice 2-3 map-specific comps rather than running the same comp everywhere.

4. **Attack/defense imbalance**: Some maps have significant attacker or defender advantages. Understanding which side you are favored on helps set realistic expectations (e.g., "We need to hold 2 checkpoints on defense because attack is easy on this map").

5. **Hero pool depth**: Heroes with high map-dependent win rate variance are "map specialists." Having a player who can flex to these heroes on the right maps creates a drafting advantage.

### For Players
- Learn which maps favor your hero pool. If your best heroes have low win rates on a map, consider expanding your pool.
- On attack-favored maps, do not panic if you lose on defense. Focus on maximizing your attack round.
- Treat the first round as the highest-impact round. Come in with a practiced plan, not just "let's see what happens."